# Phylogenetic Placement of *Lactobacillus* Species Using nf-core/phyloplace

## Table of Contents
- [Introduction](#introduction)
- [Research Question & Hypothesis](#research-question--hypothesis)
- [Analysis and Methods](#analysis-and-methods)
- [Results and Visualizations](#results-and-visualizations)
- [Discussion](#discussion)
- [Future Plan](#future-plan)
- [Limitations & Caveats](#limitations--caveats)
- [References](#references)

## Introduction

This notebook guides through reproducing the phylogenetic placement of *Lactobacillus* query sequences onto a known reference tree using the **nf-core/phyloplace** workflow. The pipeline aligns sequences to a reference alignment, places them into a reference phylogeny, and produces outputs ready for visualization.

*Lactobacilli* are bacteria that colonize human and animal body sites such as the digestive tract and female genital tract. They are among the most common probiotics found in food products like yogurt.

The **nf-core/phyloplace** pipeline, developed with Nextflow, follows bioinformatics best practices to perform phylogenetic placement using EPA-NG. It utilizes Docker or Singularity containers, ensuring easy setup and enabling high reproducibility of results.

To demonstrate the utility of the nf-core/phyloplace pipeline, this project performed a phylogenetic analysis of three long-read sequences from public genome samples of the *Lactobacillus* genus.

## Requirements

- **Nextflow installed**:
```bash
wget -qO nextflow https://github.com/nextflow-io/nextflow/releases/download/v22.10.0/nextflow
chmod +x nextflow
./nextflow -version
```
- **Docker** for container execution
- **Prepared input files**:
  - Reference MSA
  - Reference tree
  - Query FASTA sequences

## Analysis and Methods




### Step 1: Genomic Data Search
Using the command, three FASTA files of *Lactobacillus* genomes generated from the Illumina sequencing platform were downloaded.
```bash
wget ftp://ftp.sra.ebi.ac.uk/vol1/fastq/ERR485/ERR485020/ERR485020.fastq.gz
```

### Step 2: Genome Annotation Using Prokka

Prokka Command:
```bash
prokka --outdir PROKKA_run2 /Users/ajy_25yahoo.com/Desktop/Advance_BioInformatic_Project_Test/fastq/SRR9860122_unique.fasta
```


The `nf-core/phyloplace` pipeline performs phylogenetic placement of your query sequences onto a reference tree. To do this meaningfully, it often requires annotated reference sequences or properly processed reference data.


 Prokka  was run on reference bacterial genome sequences to produce a high-quality annotated `.fna` file containing nucleotide sequences of predicted genes. This annotated `.fna` file (`PROKKA_06042025.fna`) which served as reference sequences (`--refseqfile`).



### Step 3: Download nf-core/phyloplace Pipeline

The nf-core/phyloplace pipeline was downloaded using this command

```bash
nextflow run nf-core/phyloplace -r 2.0.0 -profile test,docker --outdir results/phyloplace_output -c local.config
```

### Step 4: Run Pipeline

<p>Figure: the nf-core phylo pipeline workflow</p>
<p>
<img src="images/pipeline workflow.png" width="85%" style="margin-bottom: 10px;">


Command used:

nextflow run nf-core/phyloplace -r 1.0.0 \
  --id ajy_run1 \
  --queryseqfile /Users/ajy_25yahoo.com/Desktop/Advance_BioInformatic_Project_Test/fasta/SRR9860122.fasta,/Users/ajy_25yahoo.com/Desktop/Advance_BioInformatic_Project_Test/fasta/ERR485020.fasta,/Users/ajy_25yahoo.com/Desktop/Advance_BioInformatic_Project_Test/fasta/SRR10240887.fasta \
  --refseqfile /Users/ajy_25yahoo.com/Desktop/Advance_BioInformatic_Project_Test/fastq/PROKKA_06042025/PROKKA_06042025.fna \
  --refphylogeny /Users/ajy_25yahoo.com/Desktop/Advance_BioInformatic_Project_Test/fastq/PROKKA_06042025/ref_aligned.fasta \
  --model LG+F+R6 \
  --outdir /Users/ajy_25yahoo.com/Desktop/Advance_BioInformatic_Project_Test/output/phyloplace_results \
  -profile docker


Parameters description:

nextflow run nf-core/phyloplace  	Executes the phyloplace pipeline from the nf-core collection.

  -r 1.0.0	 Specifies the release version of the pipeline (1.0.0). This ensures reproducibility by fixing the pipeline version.
  
  --id ajy_run1	  Assigns a unique identifier to run. This helps label outputs and logs.

  --queryseqfile	Comma-separated paths to the query sequences (your Lactobacillus genomes in FASTA format) to place on the reference tree.

  --refseqfile	Path to the reference sequences (in FASTA format). These should be previously annotated genomes (e.g., from PROKKA).

  --refphylogeny	Path to the reference multiple sequence alignment FASTA file. This is used to construct or support the reference phylogenetic tree.

  --model LG+F+R6	Specifies the substitution model used for phylogenetic placement. 

  -profile docker
  
  -c memory_override.config This tells Nextflow to load a custom config file that overrides default resource settings.

  
This file (memory_override.config) may contain entries like:
  process {
  withName: 'some_process' {
    cpus = 4
    memory = '8 GB'
    time = '2h'
  }
}



### Step 5: Quality Control with FastQC

Command Used
```bash
fastqc /Users/ajy_25yahoo.com/Desktop/Advance_BioInformatic_Project_Test/fastq/SRR9860122.fastq \
       /Users/ajy_25yahoo.com/Desktop/Advance_BioInformatic_Project_Test/fastq/ERR485020.fastq \
       /Users/ajy_25yahoo.com/Desktop/Advance_BioInformatic_Project_Test/fastq/SRR10240887.fastq \
       -o /Users/ajy_25yahoo.com/Desktop/Advance_BioInformatic_Project_Test/fastqc_results
---
FastQC is a widely used tool for assessing the quality of raw sequencing data. It generates detailed reports covering:

- Per-base sequence quality scores
- GC content distribution
- Adapter contamination detection
- Sequence duplication levels
- And other important quality metrics

Running FastQC **prior to downstream analyses such as genome assembly or annotation is essential to:
- Verify data quality
- Detect sequencing artifacts or biases
- Decide whether trimming or cleaning is required to improve results

### Step 6: Sequence Typing Using BLAST

In microbial genomics, Sequence Typing is a method to classify bacterial isolates into specific types or strains by comparing sequences of conserved housekeeping genes or marker regions. This classification helps to:

- Understand the strain or lineage of your bacterial sample
- Track epidemiology and outbreaks
- Investigate evolutionary relationships
- Compare your isolate to known reference types in public databases


BLAST+ Command 

The following command runs a nucleotide BLAST (`blastn`) search of the sample sequence against a local **16S ribosomal RNA** database to identify closely related sequences and potential sequence types.

```bash
blastn -query fasta/SRR9860122.fasta \
       -db 16S_ribosomal_RNA \
       -out seqtype_SRR9860122.txt \
       -outfmt "6 qseqid sseqid pident length evalue bitscore stitle" \
       -max_target_seqs 5 \
       -num_threads 2

---


## Results and Visualizations
<p>Image 1: Nextflow workflow report of the nf-core phylo pipeline run</p>
<p>
<img src="images/nf_core_phylo.png" width="85%" style="margin-bottom: 10px;">
<p>Image 2: Nextflow workflow report of the nf-core-phylo pipeline run</p>
<img src="images/nf_core_phylo2.png" width="85%" style="margin-bottom: 10px;">

<p>Image 3: Genome Annotation results using Prokka </p>
<img src="images/Prokka_aanotation.png" width="85%" style="margin-bottom: 10px;">

<p>Image 3: MLST using BLAST+ </p>
<img src="images/MLST_BLAST.png" width="85%" style="margin-bottom: 10px;">

<p>Image 4: MLST summary </p>
<img src="images/MLST_summary.png" width="85%" style="margin-bottom: 10px;">

<p>Image 3: FASTQC report </p>
<img src="images/FASTQC .png" width="85%" style="margin-bottom: 10px;">



### FastQC Summary: ERR485020.fastq
| Module | Status |
|--------|--------|
| Basic Statistics | PASS |
| Per base sequence quality | PASS |
| Per sequence quality scores | PASS |
| Per base sequence content | WARN |
| Sequence Duplication Levels | WARN |
| Adapter Content | PASS |

**Interpretation:** 
### FastQC Interpretation Summary

- Most quality metrics passed, indicating the data is generally of good quality.
- **Warnings**:
  - **Per base sequence content**: Slight bias in base composition at the beginning of reads. This is common with random hexamer priming and typically not a major concern unless extreme.
  - **Sequence duplication levels**: May suggest over-sequencing or the presence of highly abundant sequences (e.g., plasmids or repeats). Consider further analysis if duplication is very high.
  
*No major red flags were observed. The data is suitable for downstream analysis, although trimming or deduplication could be considered if issues persist across multiple samples.*


## Discussion
Pipeline execution failed. Troubleshooting required (e.g., memory/config/input format). However,
- Annotation with **Prokka** completed successfully.
- Sequence QC with **FastQC** confirms suitable input quality.
- Sequence Typing via **BLAST+** gave useful matches.

Project Management with GitHub

A remote GitHub repository was created to organize and track project-related code, pipeline configurations, input/output files, and documentation. Version control through GitHub ensured reproducibility, collaboration readiness, and transparent project evolution.


## Future Plan
- Re-run pipeline with revised config
- Check input formats and version compatibility
- Compare with tools like **IQ-TREE**, **BEAST**, **PhyloPhlAn**
- Perform extended analysis: AMR genes, virulence factors

## Limitations & Caveats
- Pipeline execution failed; no final phylogenetic tree was produced.
- Only 3 samples used, limiting analysis power.
- nf-core/phyloplace lacks extensive peer-reviewed use.

## References
- Ewels et al. (2020). *The nf-core framework*. Nature Biotechnology. https://doi.org/10.1038/s41587-020-0439-x
- Petit & Read (2020). *Bactopia pipeline*. mSystems. https://doi.org/10.1128/mSystems.00190-20
- nf-core/phyloplace GitHub: https://github.com/nf-core/phyloplace
